# Module 5 - Yield Analysis

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


## Q1 - What is the average yield per stage accross all batches?

In [2]:
avg_yield_per_stage_query = """
    SELECT
        f.stage,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    WHERE f.yield_pct IS NOT NULL
    AND f.stage NOT IN ('wet_granulation')
    GROUP BY f.stage
    ORDER BY avg_yield_pct ASC;
"""

df_avg_yield_stage = pd.read_sql_query(avg_yield_per_stage_query, engine)
df_avg_yield_stage

,stage,avg_yield_pct,batch_count
0,encapsulation,96.39,174
1,coating,96.93,696
2,compression,97.77,854
3,dry_blending,98.96,1064


In [12]:
stage_labels = {
    'dry_blending': 'Dry Blending',
    'compression': 'Compression',
    'coating': 'Coating',
    'encapsulation': 'Encapsulation'
}

df_avg_yield_stage['stage_label'] = df_avg_yield_stage['stage'].map(stage_labels)

fig = px.bar(
    df_avg_yield_stage,
    x='stage_label',
    y='avg_yield_pct',
    title='Average Yield per Stage — All Batches (2025)',
    labels={
        'avg_yield_pct': 'Average Yield (%)',
        'stage_label': ''
    },
    text='avg_yield_pct',
    color='avg_yield_pct',
    color_continuous_scale='Blues'
)

fig.add_hline(
    y=96,
    line_dash='dash',
    line_color='red',
    annotation_text='Min Target: 96%',
    annotation_position='top right'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, title='Average Yield (%)', range=[94, 100])
)
fig.show()

## Q1 — Average Yield per Stage
 
**Observation:**
Dry Blending leads at 98.96%, followed by Compression (97.77%) and Coating (96.93%). Encapsulation is the lowest-performing stage at 96.39% — the only stage that sits closest to the minimum target threshold.
 
**Insight:**
All four stages are above the minimum 96% target, which means the facility is technically compliant at the aggregate level. However, the spread between Dry Blending (98.96%) and Encapsulation (96.39%) is 2.57 percentage points — a significant gap when translated into absolute units across 1,064 batches. Encapsulation's proximity to the target floor leaves very little room for individual batch variation before breaches occur.
 
**Recommendation:**
Do not interpret facility-wide target compliance as an absence of risk. Encapsulation's 96.39% average means that any month with below-average performance will push it below the 97% target. Establish a tighter internal alert threshold of 96.5% for encapsulation to provide early warning before the official target is breached.

## Q2 - What is the average yield per dosage form per stage?

In [16]:
avg_yield_dosage_stage_query = """
    SELECT
        p.dosage_form,
        f.stage,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    WHERE f.yield_pct IS NOT NULL
    AND f.stage NOT IN ('wet_granulation')
    GROUP BY p.dosage_form, f.stage
    ORDER BY p.dosage_form, f.stage;
"""

df_avg_yield_dosage_stage = pd.read_sql_query(avg_yield_dosage_stage_query, engine)
df_avg_yield_dosage_stage

,dosage_form,stage,avg_yield_pct,batch_count
0,CAPSULE,encapsulation,96.39,174
1,CAPSULE,dry_blending,99.39,190
2,FILM-COATED TABLET,compression,98.02,424
3,FILM-COATED TABLET,coating,96.79,422
4,FILM-COATED TABLET,dry_blending,99.04,427
5,TABLET,compression,96.60,137
6,TABLET,dry_blending,98.10,138
7,SUSTAINED-RELEASE TABLET,compression,98.35,267
8,SUSTAINED-RELEASE TABLET,coating,97.19,265
9,SUSTAINED-RELEASE TABLET,dry_blending,99.12,267


In [14]:
monthly_yield_trend_query = """
    SELECT
        TO_CHAR(d.full_date, 'YYYY-MM') AS month,
        TO_CHAR(d.full_date, 'Mon') AS month_short,
        f.stage,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct
    FROM fact_batch_production f
    JOIN dim_date d ON f.date_id = d.date_id
    WHERE f.yield_pct IS NOT NULL
    AND f.stage NOT IN ('wet_granulation')
    AND d.year = 2025
    GROUP BY TO_CHAR(d.full_date, 'YYYY-MM'), TO_CHAR(d.full_date, 'Mon'), f.stage
    ORDER BY month, f.stage;
"""

df_monthly_yield_trend = pd.read_sql_query(monthly_yield_trend_query, engine)
df_monthly_yield_trend

,month,month_short,stage,avg_yield_pct
0,2025-01,Jan,compression,98.33
1,2025-01,Jan,coating,97.42
2,2025-01,Jan,encapsulation,95.34
3,2025-01,Jan,dry_blending,99.05
4,2025-02,Feb,compression,97.42
5,2025-02,Feb,coating,96.93
6,2025-02,Feb,encapsulation,95.74
7,2025-02,Feb,dry_blending,98.79
8,2025-03,Mar,compression,97.78
9,2025-03,Mar,coating,97.16


In [17]:
stage_labels = {
    'dry_blending': 'Dry Blending',
    'compression': 'Compression',
    'coating': 'Coating',
    'encapsulation': 'Encapsulation'
}

df_avg_yield_dosage_stage['stage_label'] = df_avg_yield_dosage_stage['stage'].map(stage_labels)

fig = px.bar(
    df_avg_yield_dosage_stage,
    x='dosage_form',
    y='avg_yield_pct',
    color='stage_label',
    barmode='group',
    title='Average Yield by Dosage Form and Stage (2025)',
    labels={
        'avg_yield_pct': 'Average Yield (%)',
        'dosage_form': '',
        'stage_label': 'Stage'
    },
    text='avg_yield_pct',
    color_discrete_sequence=px.colors.sequential.Blues[2:]
)

fig.add_hline(
    y=96,
    line_dash='dash',
    line_color='red',
    annotation_text='Min Target: 96%',
    annotation_position='top right'
)

fig.update_traces(textposition='outside', textfont_size=9)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, tickangle=-30),
    yaxis=dict(showgrid=False, title='Average Yield (%)', range=[92, 101]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## Q2 — Average Yield by Dosage Form and Stage
 
**Observation:**
Several dosage form-stage combinations fall below the 96% minimum target. Bilayer Tablet in the coating stage is the worst at 91.85%, followed by Enteric-Coated Tablet in coating (93.28%) and compression (95.05%). Extended Release Tablet coating sits at 95.75%. Capsule encapsulation is at 96.39% — marginally above the floor. All dry blending yields are strong across all dosage forms, ranging from 98.03% to 99.39%.
 
**Insight:**
The coating stage is the most problematic across dosage forms — specifically for Bilayer Tablet and Enteric-Coated Tablet. Bilayer Tablet's 91.85% coating yield is the single worst data point in the entire yield analysis, nearly 4.5 percentage points below the 96% target. This is consistent with the manufacturing reality that bilayer tablets have more complex compression geometry that can affect coating surface uniformity. Enteric-Coated Tablet's compression yield of 95.05% is also concerning — it is below target at the compression stage before even reaching the coating stage, suggesting cumulative loss.
 
**Recommendation:**
Prioritize Bilayer Tablet coating and Enteric-Coated Tablet compression for immediate process review. For Bilayer Tablet coating, investigate pan loading, spray rate, and coating solution parameters. For Enteric-Coated Tablet compression, examine tooling condition and compression force settings. These two dosage form-stage combinations represent the highest-risk yield gaps in the facility.


## Q3 - How does yield trend monthly across stages?

In [19]:
monthly_yield_trend_query = """
    SELECT
        TO_CHAR(d.full_date, 'YYYY-MM') AS month,
        TO_CHAR(d.full_date, 'Mon') AS month_short,
        f.stage,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct
    FROM fact_batch_production f
    JOIN dim_date d ON f.date_id = d.date_id
    WHERE f.yield_pct IS NOT NULL
    AND f.stage NOT IN ('wet_granulation')
    AND d.year = 2025
    GROUP BY TO_CHAR(d.full_date, 'YYYY-MM'), TO_CHAR(d.full_date, 'Mon'), f.stage
    ORDER BY month, f.stage;
"""

df_monthly_yield_trend = pd.read_sql_query(monthly_yield_trend_query, engine)
df_monthly_yield_trend

,month,month_short,stage,avg_yield_pct
0,2025-01,Jan,compression,98.33
1,2025-01,Jan,coating,97.42
2,2025-01,Jan,encapsulation,95.34
3,2025-01,Jan,dry_blending,99.05
4,2025-02,Feb,compression,97.42
5,2025-02,Feb,coating,96.93
6,2025-02,Feb,encapsulation,95.74
7,2025-02,Feb,dry_blending,98.79
8,2025-03,Mar,compression,97.78
9,2025-03,Mar,coating,97.16


In [18]:
stage_labels = {
    'dry_blending': 'Dry Blending',
    'compression': 'Compression',
    'coating': 'Coating',
    'encapsulation': 'Encapsulation'
}

df_monthly_yield_trend['stage_label'] = df_monthly_yield_trend['stage'].map(stage_labels)

fig = px.line(
    df_monthly_yield_trend,
    x='month_short',
    y='avg_yield_pct',
    color='stage_label',
    title='Monthly Yield Trend by Stage (2025)',
    labels={
        'avg_yield_pct': 'Average Yield (%)',
        'month_short': '',
        'stage_label': 'Stage'
    },
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.add_hline(
    y=96,
    line_dash='dash',
    line_color='red',
    annotation_text='Min Target: 96%',
    annotation_position='top right'
)

fig.update_traces(marker=dict(size=7))
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgrey',
               title='Average Yield (%)', range=[92, 101]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## Q3 — Monthly Yield Trend by Stage
 
**Observation:**
Dry Blending (pink) is the most stable line throughout the year, consistently hovering between 98.5% and 99.2% with minimal fluctuation. Compression (green) is similarly stable, ranging from 97.5% to 98.3%. Coating (orange) shows moderate variability, dipping to its lowest in June at approximately 96.3% before recovering. Encapsulation (blue) is the most volatile stage — dropping below the 96% target line in January, May, August, and September, with its lowest point in August at approximately 95.3%.
 
**Insight:**
Encapsulation is the only stage that regularly breaches the minimum 96% target, and its dips are not random — they cluster in high-volume months (August is your Q3 peak). This suggests that when encapsulation volume increases, yield degrades, which is consistent with the earlier finding that PHARMAFILL — the higher-frequency encapsulation machine — underperforms relative to NJP 3800. Higher throughput pressure likely reduces setup quality and increases fill weight variability.
 
**Recommendation:**
Implement a monthly encapsulation yield monitoring trigger — if the monthly average drops below 96.5%, initiate an immediate machine inspection before the month closes. Cross-reference the months where encapsulation yield dipped below target with PHARMAFILL's batch log to confirm whether those dips are machine-specific. Consider capping PHARMAFILL's monthly batch load during Q3 to protect yield performance during peak months.

## Q4 - Which bathes have yield below target?

In [20]:
below_target_yield_query = """
    WITH targets AS (
        SELECT
            t.stage,
            t.dosage_form,
            t.target_yield_pct / 100.0 AS target_yield
        FROM dim_stage_target t
        WHERE t.effective_date = (
            SELECT MAX(t2.effective_date)
            FROM dim_stage_target t2
            WHERE t2.stage = t.stage
            AND t2.dosage_form = t.dosage_form
        )
    )
    SELECT
        f.stage,
        COUNT(f.fact_id) AS total_batches,
        SUM(CASE WHEN f.yield_pct < t.target_yield THEN 1 ELSE 0 END) AS below_target_count,
        ROUND(
            SUM(CASE WHEN f.yield_pct < t.target_yield THEN 1 ELSE 0 END) * 100.0
            / COUNT(f.fact_id), 1
        ) AS below_target_pct
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    JOIN targets t ON f.stage = t.stage AND p.dosage_form = t.dosage_form
    WHERE f.yield_pct IS NOT NULL
    AND f.stage NOT IN ('wet_granulation')
    GROUP BY f.stage
    ORDER BY below_target_pct DESC;
"""

df_below_target = pd.read_sql_query(below_target_yield_query, engine)
df_below_target

,stage,total_batches,below_target_count,below_target_pct
0,encapsulation,174,71,40.8
1,coating,696,173,24.9
2,compression,854,199,23.3


In [21]:
stage_labels = {
    'dry_blending': 'Dry Blending',
    'compression': 'Compression',
    'coating': 'Coating',
    'encapsulation': 'Encapsulation'
}

df_below_target['stage_label'] = df_below_target['stage'].map(stage_labels)

fig = px.bar(
    df_below_target,
    x='stage_label',
    y='below_target_pct',
    title='Percentage of Batches Below Target Yield per Stage (2025)',
    labels={
        'below_target_pct': '% of Batches Below Target',
        'stage_label': ''
    },
    text='below_target_pct',
    color='below_target_pct',
    color_continuous_scale='Reds'
)

fig.update_traces(textposition='outside', texttemplate='%{text}%')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, title='% of Batches Below Target', range=[0, 50])
)
fig.show()


## Q4 — Percentage of Batches Below Target Yield per Stage
 
**Observation:**
Encapsulation has the highest proportion of below-target batches at 40.8% — meaning 4 in every 10 encapsulation batches fails to meet the 97% target. Coating follows at 24.9% and Compression at 23.3%. Dry Blending is excluded from this chart as it does not appear — its below-target rate is negligible.
 
**Insight:**
A 40.8% below-target rate for encapsulation is a critical finding. At the aggregate level, encapsulation's average yield of 96.39% looks acceptable — but the batch-level distribution tells a very different story. Nearly half of all encapsulation batches are failing to meet target, which means the acceptable batches are masking a large population of underperforming ones. The same pattern applies to coating and compression at roughly 1 in 4 batches — these are not outlier events, they are systemic.
 
**Recommendation:**
Shift the encapsulation conversation from average yield monitoring to batch-level compliance tracking. Reporting only the monthly average conceals that 40.8% of batches are non-compliant. Present batch-level below-target rates to management alongside averages to give a complete picture. For coating and compression, investigate whether the ~24% below-target rate is concentrated in specific machines or specific products — the earlier machine yield analysis suggests PHARMAFILL, KEVIN 36, and FLUIDPACK D-27 are the likely drivers.